In [0]:
"""

===============================================================================
Procedure: Load Silver Table for Sales information (Bronze -> Silver)
===============================================================================
Script Purpose:
    This stored procedure performs the ETL (Extract, Transform, Load) process to 
    populate the 'silver' schema tables from the 'bronze' schema.
	Actions Performed:to full refresh the ETL patterns
		- Truncates Silver tables if already there.
		- Inserts transformed and cleansed data from Bronze into Silver tables.
	Objectives:
		1-Validtate and convert date column
		2-Fix incorrect sales amount
		3-Handle NUll sales price 
		4-Ensure data consitencies like format validation e.g order date should be in the date format , sales price should be in int, float 

"""

In [0]:
#init 

catalog_name = "abhi_dwh_sql_based"
source_schema  = "bronze"
sink_schema = "silver"
table_name = "crm_sales_details"


# Read the sales_info from the bronze layer 

In [0]:
df = spark.read.table(f"{catalog_name}.{source_schema}.{table_name}")



In [0]:
df.show(5)

## Transformation to clean the data

In [0]:
#import necessory library 

from pyspark.sql.functions import col, row_number, replace, regexp_replace, substring, length, coalesce, lit, upper, when, trim
from pyspark.sql.window import Window 

### 0. Trim all the leading and trailing space from all the columns 

In [0]:
df = df.select(*[trim(col(c)).alias(c) if dict(df.dtypes)[c] == 'string' else col(c) for c in df.columns])

### 1.Validate and convert the date columns

In [0]:
df.dtypes


In [0]:
#Based on above results date columns are in the int dtypes 

date_columns = []
for column_name in df.columns:
    if "dt" in column_name.split("_"):
        date_columns.append(column_name)
print(date_columns)

In [0]:
#Date columns are in format like yyyymmdd

#convert the date columns to date format and drop the original columns 
from pyspark.sql.functions import expr

for column_name in date_columns:
    df = df.withColumn(column_name, regexp_replace(column_name, r"(\d{4})(\d{2})(\d{2})", r"$1-$2-$3"))
    # df = df.withColumn(column_name, col(column_name).try_cast("date")) This was creating the issue while writting the data into the silver table issue is not able to convert the date columns
    df = df.withColumn(column_name, expr(f"try_cast(`{column_name}` as date)"))
    # df = df.drop(column_name)
df.show(5)


In [0]:
df.dtypes

In [0]:
# df.filter(col("sls_due_dt")==0).count()

### 2.Fix Incorrect sales amount 

In [0]:
print(df.filter(col("sls_sales") == 0).count())
print(df.filter(col("sls_sales").isNull()).count())

In [0]:
#1st will make the consitent values like 0's into null, it will easy to maintain the data and code 

df = df.withColumn("sls_sales" , when(col("sls_sales")==0, None).otherwise(col("sls_sales")))

In [0]:
df.filter(col("sls_quantity") >=2).show(5)

In [0]:
#Since there are chances of the sales being zero. After discussion with the stake holder and expert we came to know that the sls_sales = sls_price x sls_quantity 
#Now we will replace the null values with the priduct of price and quantity 

df = df.withColumn("sls_sales", coalesce(col("sls_sales"), col("sls_price") * col("sls_quantity")))
df.show(5)


In [0]:
print(df.filter(col("sls_sales") == 0).count())
print(df.filter(col("sls_sales").isNull()).count())

### 3.Fix invalid prices


In [0]:
df.filter((col("sls_price").isNull()) | (col("sls_price") == 0)).count()

In [0]:
#Since there are chances of the sales being zero. After discussion with the stake holder and expert we came to know that the sls_sales = sls_price x sls_quantity

#it meanse we can make the sls_price = sls_sales / sls_quantity

df = df.withColumn("sls_price", coalesce(col("sls_price"), col("sls_sales") / col("sls_quantity")))
df.show(5)

In [0]:
df.filter((col("sls_price").isNull()) | (col("sls_price") == 0)).count()

# Write it to Silver Layer after cleansing 

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{sink_schema}.{table_name}")


In [0]:
df.show()